# Test ASR — audio chưa có transcript

Notebook độc lập để nghe audio và xem text model dự đoán, tương tự cell test text trong notebook `01_train_asr_bud500.ipynb`. **Không train, không cập nhật weights, không tính/hiển thị loss, CER hoặc WER.**

- Load **best epoch** từ `training_runs/ASR_BUD500_2X_RANDOM42_TEST20/best.pt`, không load `last.pt`.
- Đọc đệ quy `data/ASR/testASR/`, bao gồm thư mục con `test/`. Lúc tạo notebook có **857 WAV**, mono 16 kHz, dài khoảng 4,98–38,10 giây.
- Mặc định lấy **15 file ngẫu nhiên, không lặp**, seed 42; chạy từng file để hạn chế VRAM.
- Mỗi case hiển thị tên file, text dự đoán, trạng thái và nút nghe audio.
- Không có transcript chuẩn: **không thể đo accuracy/CER/WER**. `status=ok` chỉ nghĩa decode thành công, không chứng minh text đúng. Dự đoán không được tự động ghi thành transcript/ground truth.
- Không sửa audio nguồn, checkpoint, notebook train hoặc manifest. Chưa kiểm tra bộ audio này có trùng dữ liệu train hay không; đây là kiểm tra định tính, không phải benchmark độc lập đã xác minh.

Chọn kernel Python có `torch`, `torchaudio`, `soundfile`, `numpy`, `ipykernel`, rồi **Run All**. Không cần chạy notebook `01` trước. Nếu CUDA hết bộ nhớ, đóng model ở notebook khác hoặc đổi `device` sang `cpu`; không cắt audio âm thầm.

In [1]:
TEST = {
    'audio_dir': 'data/ASR/testASR',
    'checkpoint': 'training_runs/ASR_BUD500_2X_RANDOM42_TEST20/best.pt',
    'num_cases': 15,
    'sample_seed': 42,  # Đổi seed hoặc đặt None để mỗi lần chọn bộ mẫu khác.
    'device': 'auto',  # auto / cuda / cpu
    'show_audio': True,
    'enabled': True,   # Chỉ inference; không có đường train trong notebook này.
}


In [2]:
import gc, random, sys, warnings
from pathlib import Path
from collections import Counter
import torch
import torchaudio
import soundfile as sf
from IPython.display import Audio, display

def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'model/ASR/model.py').is_file() and (candidate / 'data/ASR').is_dir():
            return candidate
    raise FileNotFoundError('Không tìm thấy project root; mở notebook trong thư mục ModelForValls')

ROOT = find_project_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from model.ASR.model import load_for_inference, load_audio

if TEST['device'] not in ('auto', 'cuda', 'cpu'):
    raise ValueError('device phải là auto, cuda hoặc cpu')
if TEST['device'] == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('Kernel hiện tại không có CUDA; chọn device=cpu hoặc đổi kernel')
DEVICE = torch.device(('cuda' if torch.cuda.is_available() else 'cpu') if TEST['device'] == 'auto' else TEST['device'])
AUDIO_ROOT = (ROOT / TEST['audio_dir']).resolve()
CHECKPOINT_PATH = (ROOT / TEST['checkpoint']).resolve()
print('Python:', sys.executable)
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(DEVICE))
if torch.__version__.split('+')[0].split('.')[:2] != torchaudio.__version__.split('+')[0].split('.')[:2]:
    warnings.warn('Torch/torchaudio khác release; nên dùng cặp phiên bản chính thức tương thích')


Python: c:\Users\ASUS\AppData\Local\Programs\Python\Python313\python.exe
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


C:\Users\ASUS\AppData\Local\Temp\ipykernel_12020\1671825263.py:32: UserWarning: Torch/torchaudio khác release; nên dùng cặp phiên bản chính thức tương thích
  warnings.warn('Torch/torchaudio khác release; nên dùng cặp phiên bản chính thức tương thích')


## Chọn 15 audio ngẫu nhiên
Danh sách file được sort trước khi lấy mẫu để seed có thể tái lập. Mỗi file được chọn tối đa một lần; không đảm bảo nội dung của hai file khác tên là khác nhau. Nếu thư mục có ít hơn 15 file sẽ dùng tất cả file hiện có.

Để lấy 15 mẫu khác: đổi `TEST['sample_seed']` rồi chạy lại cell chọn mẫu và cell dự đoán cuối. Không cần load dataset có nhãn hay chuẩn bị manifest. Nếu thay `audio_dir`, `checkpoint` hoặc `device`, chạy lại cell setup.

In [3]:
if not AUDIO_ROOT.is_dir():
    raise FileNotFoundError(f'Không có thư mục audio: {AUDIO_ROOT}')
if type(TEST['num_cases']) is not int or TEST['num_cases'] < 1:
    raise ValueError('num_cases phải là số nguyên dương')
if TEST['sample_seed'] is not None and type(TEST['sample_seed']) is not int:
    raise ValueError('sample_seed phải là số nguyên hoặc None')
AUDIO_EXTENSIONS = {'.wav', '.flac', '.ogg', '.aiff', '.aif'}
AUDIO_FILES = sorted((p for p in AUDIO_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS),
                     key=lambda p: p.relative_to(AUDIO_ROOT).as_posix())
if not AUDIO_FILES:
    raise ValueError(f'Không tìm thấy audio WAV/FLAC/OGG/AIFF trong {AUDIO_ROOT}')
SELECTED_AUDIO = random.Random(TEST['sample_seed']).sample(AUDIO_FILES, min(TEST['num_cases'], len(AUDIO_FILES)))
print(f'Tìm thấy {len(AUDIO_FILES)} file; chọn {len(SELECTED_AUDIO)} case, seed={TEST["sample_seed"]}')
for index, path in enumerate(SELECTED_AUDIO, 1):
    print(f'{index:02d}. {path.relative_to(AUDIO_ROOT).as_posix()}')


Tìm thấy 857 file; chọn 15 case, seed=42
01. test/5714031489489416595.wav
02. test/12294378439351664917.wav
03. test/10531614475822620460.wav
04. test/7933565751216072063.wav
05. test/15475286645831936233.wav
06. test/15048078677738670465.wav
07. test/1458760945721625996.wav
08. test/12858037995647160612.wav
09. test/7891017784103763965.wav
10. test/12103061109919769542.wav
11. test/6324265247704115704.wav
12. test/792774605292323776.wav
13. test/4051886311694235581.wav
14. test/11765686285773461217.wav
15. test/4856580170092767121.wav


## Text dự đoán từ best epoch
Cell cuối load config + weights của `best.pt`, hiển thị số epoch theo cách đếm **từ 1**, và chạy `transcribe()` ở chế độ inference/eval. Audio được đọc qua frontend chung của model: PCM float32, mono, resample thật về sample rate trong checkpoint; **không crop, không pad/kéo dài hoặc nối transcript**.

Nghe từng audio để tự đối chiếu text. Lỗi đọc audio, thiếu bộ nhớ, UTF-8 không hợp lệ hoặc text rỗng được hiển thị riêng và giữ trong `TEST_TEXT_RESULTS`; không thay bằng transcript bịa và không âm thầm bỏ case. Tổng hợp cuối cell chỉ đếm trạng thái, không phải accuracy. Các widget phát audio không tự chuẩn hóa biên độ.

Kết quả nằm trong biến `TEST_TEXT_RESULTS` của kernel; không ghi đè file nguồn, checkpoint hay tạo nhãn train. Model được giải phóng sau mỗi lần chạy cell.

In [4]:
TEST_TEXT_RESULTS = []
if TEST['enabled']:
    if not CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(f'Chưa có best checkpoint: {CHECKPOINT_PATH}')
    if not SELECTED_AUDIO:
        raise ValueError('Chạy cell chọn audio trước khi dự đoán')
    checkpoint_metadata = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
    try:
        epoch_index = checkpoint_metadata.get('epoch')
        if type(epoch_index) is not int or epoch_index < 0:
            raise ValueError('Checkpoint không có epoch đã train hợp lệ')
        BEST_EPOCH = epoch_index + 1
    finally:
        del checkpoint_metadata
    best_model = load_for_inference(CHECKPOINT_PATH, DEVICE)
    print(f'Best epoch: {BEST_EPOCH} | {len(SELECTED_AUDIO)} case | device={DEVICE}')
    print('Checkpoint:', CHECKPOINT_PATH)
    try:
        for case_number, audio_path in enumerate(SELECTED_AUDIO, 1):
            relative_path = audio_path.relative_to(AUDIO_ROOT).as_posix()
            waveform = None
            duration_seconds = None
            prediction = None
            try:
                waveform, sample_rate = load_audio(audio_path, target_rate=best_model.config['audio']['sample_rate'])
                duration_seconds = waveform.numel() / sample_rate
            except (OSError, RuntimeError, ValueError, TypeError) as error:
                prediction = {'text': None, 'status': 'audio_read_error', 'error': str(error)}
            if prediction is None:
                try:
                    # Chỉ đưa waveform vào model; dataset này không có transcript chuẩn.
                    prediction = best_model.transcribe(waveform, sample_rate, request_id=relative_path)
                except torch.OutOfMemoryError:
                    prediction = {'text': None, 'status': 'out_of_memory',
                                  'error': 'Không đủ bộ nhớ; đóng model ở notebook khác hoặc chọn device=cpu rồi chạy lại setup'}
                    if DEVICE.type == 'cuda':
                        torch.cuda.empty_cache()
                except (RuntimeError, ValueError) as error:
                    prediction = {'text': None, 'status': 'inference_error', 'error': str(error)}
            result = {'case': case_number, 'audio_file': relative_path, 'duration_seconds': duration_seconds,
                      'reference_text': None, 'predicted_text': prediction['text'],
                      'status': prediction['status'], 'best_epoch': BEST_EPOCH}
            if prediction.get('error'):
                result['error'] = prediction['error']
            TEST_TEXT_RESULTS.append(result)
            print('\n' + '=' * 90)
            print(f'Case {case_number:02d}/{len(SELECTED_AUDIO)} | {relative_path}')
            if duration_seconds is not None:
                print(f'Thời lượng     : {duration_seconds:.2f} giây')
            print('Transcript gốc : chưa có')
            predicted_text = prediction['text']
            if predicted_text is None:
                predicted_text = f'<không có text hợp lệ: {prediction["status"]}>'
            elif not predicted_text.strip():
                predicted_text = '<text rỗng>'
            print('Model dự đoán  :', predicted_text)
            print('Trạng thái     :', prediction['status'])
            if prediction.get('error'):
                print('Chi tiết       :', prediction['error'])
            if TEST['show_audio'] and waveform is not None:
                try:
                    display(Audio(waveform.detach().cpu().numpy(), rate=sample_rate, normalize=False))
                except (ValueError, RuntimeError, OSError) as error:
                    print('Không hiển thị được player; mở file nguồn để nghe:', error)
            del waveform
        print('\nSố case theo trạng thái (không phải accuracy):', dict(Counter(r['status'] for r in TEST_TEXT_RESULTS)))
    finally:
        del best_model
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
else:
    print('Inference đang tắt; đặt TEST["enabled"] = True rồi chạy lại cell này.')


Best epoch: 13 | 15 case | device=cuda
Checkpoint: D:\ModelForValls\training_runs\ASR_BUD500_2X_RANDOM42_TEST20\best.pt

Case 01/15 | test/5714031489489416595.wav
Thời lượng     : 15.12 giây
Transcript gốc : chưa có
Model dự đoán  : cao ơghi ra hay khô lăng màu ghi ra trong tuâm lống lăng màt cá ai cạp có nhiều khim thự thạp trong đó đại kiểu thời thạn là kinh thự tháp lớp nhất nhiều màu nhó nhiều đền đày và từng đạy nhân sứ 
Trạng thái     : ok


Invalid UTF-8 CTC path for request test/12294378439351664917.wav



Case 02/15 | test/12294378439351664917.wav
Thời lượng     : 13.08 giây
Transcript gốc : chưa có
Model dự đoán  : <không có text hợp lệ: invalid_utf8>
Trạng thái     : invalid_utf8



Case 03/15 | test/10531614475822620460.wav
Thời lượng     : 15.96 giây
Transcript gốc : chưa có
Model dự đoán  : năm hai hàn lẽ hai thành phố go ma bị xem bầm bởn nỳ lửa ni ghe gon cô thảm hạ nố lỡ phong chào này đã chon vồi phần lớn thành phố này đặc biệt là khô vột trung tâm thành phố 
Trạng thái     : ok



Case 04/15 | test/7933565751216072063.wav
Thời lượng     : 20.88 giây
Transcript gốc : chưa có
Model dự đoán  : vung quốc ma rốt đã xây dựng lại thçn phố này giống như đau lun bắt đa và nó được đặt tên là ca xa lanh ca bởi cát thương nhân tây ba nha là những người đã lập các cơ sở giao thư ß đó 
Trạng thái     : ok


...

Invalid UTF-8 CTC path for request test/11765686285773461217.wav



Case 14/15 | test/11765686285773461217.wav
Thời lượng     : 15.42 giây
Transcript gốc : chưa có
Model dự đoán  : <không có text hợp lệ: invalid_utf8>
Trạng thái     : invalid_utf8


Invalid UTF-8 CTC path for request test/4856580170092767121.wav



Case 15/15 | test/4856580170092767121.wav
Thời lượng     : 12.54 giây
Transcript gốc : chưa có
Model dự đoán  : <không có text hợp lệ: invalid_utf8>
Trạng thái     : invalid_utf8



Số case theo trạng thái (không phải accuracy): {'ok': 7, 'invalid_utf8': 8}
